## Next-Character Prediction (RNN)


In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 1. Data Preparation
# 1-1. Define word list
fruits = ["apple", "banana", "cherry", "orange", "grape", "mango", "peach", "melon", "kiwi", "lemon"]
data = [f"{word}" for word in fruits]

# 1-2. Create Vocabulary
all_chars = sorted(list(set("".join(data))))
char_to_idx = {ch: i for i, ch in enumerate(all_chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(char_to_idx)

# 2. Print 10 fruit names before training
print("--- Training Data (10 fruits) ---")
print(fruits)
print("-" * 35)
print(f"Vocabulary: {''.join(all_chars)}")
print(f"Vocabulary Size: {vocab_size}")
print("-" * 35)

# 3. Define RNN Model
class NextCharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(NextCharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=False)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        rnn_out, hidden = self.rnn(x, hidden)
        output = self.fc(rnn_out[-1]) 
        return output, hidden

    def init_hidden(self):
        # Initialize hidden state with zeros
        return torch.zeros(self.num_layers, 1, self.hidden_size)

# Set Hyperparameters
input_size = vocab_size
hidden_size = 20
output_size = vocab_size
learning_rate = 0.005
num_layers = 1

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("-" * 35)

model = NextCharRNN(input_size, hidden_size, output_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# 4. Model Training
print("\n--- Start Model Training ---")
epochs = 300
for epoch in range(epochs):
    total_loss = 0
    for word in data:
        # Iterate through the characters of the word to create input/label pairs
        for i in range(1, len(word)):
            input_str = word[:i]
            label_char = word[i]
            
            # Convert characters to indices
            input_indices = [char_to_idx[ch] for ch in input_str]
            label_index = char_to_idx[label_char]
            
            # One-hot encode the input sequence
            input_onehot = np.eye(vocab_size)[input_indices]
            
            # Convert to PyTorch tensors
            inputs = torch.Tensor(input_onehot).view(len(input_str), 1, vocab_size).to(device)
            labels = torch.LongTensor([label_index]).to(device)
            
            # Initialize hidden state for each sequence
            hidden = model.init_hidden().to(device)
            optimizer.zero_grad()
            
            # Forward pass
            outputs, hidden = model(inputs, hidden)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Average Loss: {total_loss/len(data):.4f}")
print("-" * 35)

# 5. Define prediction function based on user input (★★ Modified Version ★★)
def predict(model, start_str):
    """Completes the word based on the user's starting string."""
    model.eval() 
    with torch.no_grad():
        start_str = start_str.lower()
        device = next(model.parameters()).device
        
        # Convert start string to tensor
        input_indices = [char_to_idx[ch] for ch in start_str]
        input_onehot = np.eye(vocab_size)[input_indices]
        inputs = torch.Tensor(input_onehot).view(len(start_str), 1, vocab_size).to(device)
        
        predicted_word = list(start_str)
        
        # --- Start of modified logic ---

        # 1. Pass the start string through the model to get the first prediction and hidden state
        hidden = model.init_hidden().to(device)
        initial_output, hidden = model(inputs, hidden)
        
        # 2. Add the first prediction result to the word
        _, top_idx = torch.max(initial_output, 1)
        predicted_char = idx_to_char[top_idx.item()]
        
        # If the model predicts an empty character (stop token), return the current word
        if predicted_char == '':
            return "".join(predicted_word)
        
        predicted_word.append(predicted_char)
        
        # 3. Use the just-predicted character as the input for the next loop
        last_char_idx = top_idx.item()
        
        # --- End of modified logic ---

        # Repeat character generation until an empty string is predicted or max length is reached
        for _ in range(20): 
            # Prepare the next input (the last predicted character)
            input_onehot = np.eye(vocab_size)[[last_char_idx]]
            inputs = torch.Tensor(input_onehot).view(1, 1, vocab_size).to(device)
            
            # Use the updated hidden state from the previous step
            output, hidden = model(inputs, hidden)
            
            # Get the index of the highest probability character
            _, top_idx = torch.max(output, 1)
            predicted_char = idx_to_char[top_idx.item()]
            
            # Stop if the model predicts an empty character
            if predicted_char == '':
                break
            
            predicted_word.append(predicted_char)
            last_char_idx = top_idx.item()

    return "".join(predicted_word)

# 6. Get user input and run prediction
print("\n--- Word Completer ---")
print("Enter the beginning of a word (e.g., 'ba', 'ch', 'le')")
print("Type 'exit' to quit.")
while True:
    user_input = input("Input: ")
    if user_input.lower() == 'exit':
        print("Exiting the program.")
        break
    
    try:
        completed_word = predict(model, user_input)
        print(f"  -> Model prediction: {completed_word}")
    except KeyError:
        print(f"  -> Error: Some characters in '{user_input}' are not in the vocabulary. Please try other characters.")
    except IndexError:
         print(f"  -> Error: Input is empty. Please enter some characters.")


## Next-Character Prediction(LSTM)

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 1. Data Preparation (Same as before, but with <eos> token)
fruits = ["apple", "banana", "cherry", "orange", "grape", "mango", "peach", "melon", "kiwi", "lemon"]
# Add an <eos> (end of sentence) token to each word
data = [f"{word}<eos>" for word in fruits] 
# Create vocabulary including the <eos> token
all_chars = sorted(list(set("".join(data))))
char_to_idx = {ch: i for i, ch in enumerate(all_chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(char_to_idx)

print("--- Training Data (10 fruits with <eos>) ---")
print(data)
print("-" * 35)
print(f"Vocabulary: {''.join(all_chars)}")
print(f"Vocabulary Size: {vocab_size}")
print("-" * 35)


# 2. Define Model (LSTM Model)
class NextCharLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(NextCharLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # Use nn.LSTM instead of nn.RNN
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=False)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        lstm_out, hidden = self.lstm(x, hidden)
        output = self.fc(lstm_out[-1])
        return output, hidden

    def init_hidden(self):
        # LSTM hidden state is a tuple (h0, c0)
        h0 = torch.zeros(self.num_layers, 1, self.hidden_size)
        c0 = torch.zeros(self.num_layers, 1, self.hidden_size)
        return (h0, c0)

# Set Hyperparameters
input_size = vocab_size
hidden_size = 40 # Increased hidden size
output_size = vocab_size
learning_rate = 0.005
num_layers = 2 # Increased number of layers

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("-" * 35)

model = NextCharLSTM(input_size, hidden_size, output_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


# 3. Model Training
print("\n--- Start Model Training ---")
epochs = 300
for epoch in range(epochs):
    total_loss = 0
    for word in data:
        for i in range(1, len(word)):
            input_str = word[:i]
            label_char = word[i]
            
            input_indices = [char_to_idx[ch] for ch in input_str]
            label_index = char_to_idx[label_char]
            
            input_onehot = np.eye(vocab_size)[input_indices]
            
            inputs = torch.Tensor(input_onehot).view(len(input_str), 1, vocab_size).to(device)
            labels = torch.LongTensor([label_index]).to(device)
            
            # Initialize hidden state and send to device
            hidden = tuple([h.to(device) for h in model.init_hidden()])
            
            optimizer.zero_grad()
            outputs, hidden = model(inputs, hidden)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Average Loss: {total_loss/len(data):.4f}")

print("-" * 35)


# 4. Define prediction function based on user input (★★ Final Modified Version ★★)
def predict(model, start_str):
    model.eval()
    with torch.no_grad():
        start_str = start_str.lower()
        device = next(model.parameters()).device
        
        predicted_word = list(start_str)
        hidden = tuple([h.to(device) for h in model.init_hidden()])

        # If the input is empty, set the starting character to <eos> to generate a new word
        if not start_str:
            last_char_idx = char_to_idx['<eos>']
        else:
            # 1. Pass the start string through the model to get the first prediction and context (hidden)
            input_indices = [char_to_idx[ch] for ch in start_str]
            inputs = torch.Tensor(np.eye(vocab_size)[input_indices]).view(len(start_str), 1, vocab_size).to(device)
            output, hidden = model(inputs, hidden)
            
            # 2. Add the first prediction result to the word
            _, top_idx = torch.max(output, 1)
            predicted_char = idx_to_char[top_idx.item()]
            
            # If the first prediction is <eos>, exit immediately
            if predicted_char == '<eos>':
                return "".join(predicted_word)
            
            predicted_word.append(predicted_char)
            
            # 3. Use the just-predicted character as the first input for the next generation loop
            last_char_idx = top_idx.item()

        # 4. Loop to generate the rest of the word
        for _ in range(20):
            inputs = torch.Tensor(np.eye(vocab_size)[[last_char_idx]]).view(1, 1, vocab_size).to(device)
            output, hidden = model(inputs, hidden)
            
            _, top_idx = torch.max(output, 1)
            predicted_char = idx_to_char[top_idx.item()]
            
            # Stop generation if <eos> token is predicted
            if predicted_char == '<eos>':
                break
            
            predicted_word.append(predicted_char)
            last_char_idx = top_idx.item()

    return "".join(predicted_word)


# 5. Get user input and run prediction
print("\n--- Word Completer ---")
print("Enter the beginning of a word (e.g., 'ba', 'ch', 'le')")
print("Type 'exit' to quit.")

while True:
    user_input = input("Input: ")
    if user_input.lower() == 'exit':
        print("Exiting the program.")
        break
    
    try:
        completed_word = predict(model, user_input)
        print(f"  -> Model prediction: {completed_word}")
    except KeyError as e:
        print(f"  -> Error: '{e.args[0]}' is not in the vocabulary. Please try other characters.")
    except IndexError:
         print(f"  -> Error: Input is empty. Please enter some characters.")
